In [1]:
# Nama  : Deviana Azzahroh
# NIM   : 240401010127
# Kelas : IF403

import pandas as pd

In [2]:
url = "https://raw.githubusercontent.com/Nas-virat/Telco-Customer-Churn/master/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Ukuran dataset:", df.shape)
print("\nPersentase Churn:")
print(df["Churn"].value_counts(normalize=True))

df.head()

Ukuran dataset: (7043, 21)

Persentase Churn:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df["TotalCharges"] = df["TotalCharges"].str.strip()
df["TotalCharges"] = df["TotalCharges"].replace("", pd.NA)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

print("Jumlah data kosong:")
print(df.isnull().sum().sum())

Jumlah data kosong:
0


In [4]:
df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [5]:
df = pd.get_dummies(
    df,
    columns=[
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod"
    ],
    drop_first=True,
    dtype=int
)

print("Jumlah kolom setelah encoding:", df.shape[1])

df.head()

Jumlah kolom setelah encoding: 31


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,0,1,0,0,1,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,1,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0


In [6]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

print("Jumlah fitur:", X.shape[1])
print("Jumlah target:", y.shape[0])

Jumlah fitur: 30
Jumlah target: 7043


In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_training, X_test, y_training, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Data Training: {X_training.shape[0]} baris")
print(f"Data Test: {X_test.shape[0]} baris")

Data Training: 5634 baris
Data Test: 1409 baris


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

rf.fit(X_training, y_training)

print("Model Random Forest berhasil dibuat dan dilatih.")

Model Random Forest berhasil dibuat dan dilatih.


In [12]:
# Prediksi kelas
y_pred = rf.predict(X_test)

# Prediksi probabilitas churn
y_prob = rf.predict_proba(X_test)[:, 1]

print("Prediksi berhasil dilakukan.")

Prediksi berhasil dilakukan.


In [13]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

Precision : 0.5671
Recall    : 0.6551
F1-Score  : 0.6079
ROC-AUC   : 0.8275


In [14]:
print("Classification Report")
print(classification_report(y_test, y_pred))

Classification Report
              precision    recall  f1-score   support

           0       0.87      0.82      0.84      1035
           1       0.57      0.66      0.61       374

    accuracy                           0.78      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.78      0.78      1409



In [15]:
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix
[[848 187]
 [129 245]]


In [16]:
hasil = pd.DataFrame({
    "Aktual": y_test.values,
    "Prediksi": y_pred,
    "Probabilitas_Churn": y_prob
})

hasil.head(10)

,Aktual,Prediksi,Probabilitas_Churn
0,0,0,0.003333
1,0,1,0.823333
2,0,0,0.120000
3,0,0,0.463333
4,0,0,0.006667
5,0,1,0.556667
6,0,1,0.506667
7,0,0,0.153333
8,0,0,0.020000
9,1,1,0.626667


In [17]:
hasil_risiko = hasil.sort_values(
    by="Probabilitas_Churn",
    ascending=False
)

hasil_risiko.head(10)

,Aktual,Prediksi,Probabilitas_Churn
341,1,1,1.000000
1252,1,1,1.000000
1289,1,1,1.000000
618,1,1,1.000000
629,0,1,1.000000
38,1,1,0.996667
171,1,1,0.996667
813,1,1,0.993333
1178,1,1,0.993333
1109,1,1,0.986667
